**Install and Imports**

In [ ]:
!pip uninstall -y datasets
!pip install datasets==2.18.0 transformers seqeval

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification
import numpy as np
from seqeval.metrics import classification_report

**Task 1: Dataset Selection**

In [ ]:
dataset = load_dataset("conll2003")

label_list = dataset["train"].features["chunk_tags"].feature.names
print("Labels:", label_list)

**Task 2: Data Preprocessing**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)


**Task 3: Model Setup**

In [ ]:

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

In [ ]:

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)
#Task 4: Training
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no"
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

trainer.train()


**Task 5: Evaluation**

In [ ]:
predictions, labels, _ = trainer.predict(tokenized_dataset["validation"])
preds = np.argmax(predictions, axis=2)

true_labels = [
    [label_list[l] for l in label if l != -100]
    for label in labels
]

true_preds = [
    [label_list[p] for (p, l) in zip(pred, label) if l != -100]
    for pred, label in zip(preds, labels)
]

print("\nClassification Report:\n")
print(classification_report(true_labels, true_preds))


Task 6: Inference

In [ ]:
import torch

# move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

sentence = "John works at Google in California"
tokens = sentence.split()

inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

# move inputs to same device
inputs = {key: val.to(device) for key, val in inputs.items()}

# prediction
outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=2)

predicted_labels = [label_list[p.item()] for p in predictions[0]]

print("\nInference Result:")
for token, label in zip(tokens, predicted_labels):
    print(f"{token} -> {label}")

** Task 7: Comparison between POS Tagging and Chunking**
### POS Tagging
- POS (Part-of-Speech) tagging assigns grammatical labels to each word.
- Examples: Noun (NN), Verb (VB), Adjective (JJ)
- It is a **word-level classification task**.
- Helps understand the grammatical structure of a sentence.
- Example:
  - John → Noun
  - runs → Verb

### Chunking
- Chunking groups words into meaningful phrases such as noun phrases (NP) and verb phrases (VP).
- It is a **phrase-level classification task**.
- Helps in identifying sentence structure beyond individual words.
- Example:
  - "John works" → NP + VP

### Key Differences
| Aspect | POS Tagging | Chunking |
|------|------------|----------|
| Level | Word-level | Phrase-level |
| Complexity | Easier | More complex |
| Output | Grammar tags | Phrase groups |
| Use case | Syntax understanding | Structure understanding |

### Conclusion
POS tagging is simpler and focuses on individual words, whereas chunking is more advanced and focuses on grouping words into meaningful phrases.

## Task 8: Report / Blog

### Differences between POS Tagging and Chunking

Part-of-Speech (POS) Tagging and Chunking are both important tasks in Natural Language Processing, but they operate at different levels.

- **POS Tagging** assigns grammatical labels (such as noun, verb, adjective) to each individual word in a sentence. It is a word-level classification task and helps in understanding the syntactic role of each word.

- **Chunking**, also known as shallow parsing, groups words into meaningful phrases such as noun phrases (NP), verb phrases (VP), etc. It is a phrase-level task and provides a higher-level understanding of sentence structure.

**Key Difference:**  
POS tagging focuses on individual words, while chunking focuses on grouping words into phrases.

---

### Challenges Faced

- One of the main challenges was **label alignment** due to BERT’s subword tokenization.
- Words are often split into multiple tokens, so assigning correct labels required careful handling.
- Special tokens like `[CLS]` and `[SEP]` needed to be ignored using `-100`.
- Managing variable sequence lengths and padding was also challenging.
- Some rare labels had low prediction performance due to limited training samples.

---

### Observations and Insights

- The BERT model performed very effectively for token classification tasks.
- Achieved a high F1 score (~0.92), indicating strong model performance.
- Contextual embeddings helped the model understand relationships between words.
- Proper preprocessing (especially label alignment) is critical for good results.
- Even with limited epochs, the model achieved strong accuracy.

---

### Conclusion

This assignment demonstrated that transformer models like BERT are highly powerful for sequence labeling tasks such as POS tagging and chunking. With proper preprocessing and training, high accuracy can be achieved in NLP applications.